## Lesson 4: Email Assistant with Semantic + Episodic Memory

In [ ]:
# 加载 .env 环境变量
import os
from dotenv import load_dotenv
_ = load_dotenv()

## Repeat setup from previous lesson

In [ ]:
# 用户画像，同前几课
profile = {
    "name": "John",
    "full_name": "John Doe",
    "user_profile_background": "Senior software engineer leading a team of 5 developers",
}

In [ ]:
# 分诊规则 + agent 行为指令，同前几课
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage John's tasks efficiently."
}

In [ ]:
# 示例邮件（下面会被同名变量 email 覆盖成另一种 schema，用于 few-shot 示例，见后续 cell）
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "body": """
Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

 ## Look at a few, few-shot-examples

**架构说明**：本课引入**情景记忆（Episodic Memory）**——把"过去处理过的具体案例
（一封邮件 + 人工给的正确分类标签）"存进 store，之后分诊时通过语义检索把最相关的
几个历史案例作为 few-shot 示例塞进 prompt，让 LLM"参考先例"来做判断。
这和 lesson_3 的语义记忆（存事实性知识，如"Jim 是朋友"）是互补的两种长期记忆。

In [ ]:
from langgraph.store.memory import InMemoryStore

In [ ]:
# 同样配置向量索引，之后才能对存进去的"历史案例"做语义相似度检索
store=InMemoryStore(
    index={"embed": "openai:text-embedding-3-small"}
)

In [ ]:
# 注意：这里把 email 重新赋值成了另一种 schema（author/to/subject/email_thread），
# 和上面那个 from/body 结构不同——是为了配合下面 triage 的示例存储格式，属有意为之，不是笔误
email = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

In [ ]:
# 一条"情景记忆"记录：邮件本身 + 人工标注的正确分类（label）
data = {
    "email": email,
    # This is to start changing the behavior of the agent
    "label": "respond"
}

In [ ]:
# store.put(namespace, key, value)：把这条示例写入 store。
# 命名空间是 ("email_assistant", "lance", "examples")——注意末尾是 "examples"，
# 和 lesson_3 里存事实记忆用的 "collection" 是不同的命名空间，两种记忆互不干扰。
# key 用随机 uuid，因为这里只是"追加一条案例"，不需要按业务含义寻址。
# 这一步会调用 embedding API 为写入内容建索引（真实网络请求，假 key 场景下预期报错）
import uuid
store.put(
    ("email_assistant", "lance", "examples"),
    str(uuid.uuid4()),
    data
)

### Store a Second Example

In [ ]:
# 第二条情景记忆示例：这封"只是同步进展，不需要回复"的邮件，人工标注为 "ignore"
data = {
    "email": {
        "author": "Sarah Chen <sarah.chen@company.com>",
        "to": "John Doe <john.doe@company.com>",
        "subject": "Update: Backend API Changes Deployed to Staging",
        "email_thread": """Hi John,

    Just wanted to let you know that I've deployed the new authentication endpoints we discussed to the staging environment. Key changes include:

    - Implemented JWT refresh token rotation
    - Added rate limiting for login attempts
    - Updated API documentation with new endpoints

    All tests are passing and the changes are ready for review. You can test it out at staging-api.company.com/auth/*

    No immediate action needed from your side - just keeping you in the loop since this affects the systems you're working on.

    Best regards,
    Sarah
    """,
    },
    "label": "ignore"
}

In [ ]:
store.put(
    ("email_assistant", "lance", "examples"),
    str(uuid.uuid4()),
    data
)

### Simulate searching and returning examples

In [ ]:
# Template for formating an example to put in prompt
# 把一条 store 里的示例记录格式化成一段可读文本，用于拼进分诊 prompt 的 {examples} 占位符
template = """Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content:
```
{content}
```
> Triage Result: {result}"""

# Format list of few shots
# examples 是 store.search() 返回的结果列表，每一项 eg 是一个 Item，
# 用 eg.value 拿到当初 store.put() 存的原始 dict（{"email":..., "label":...}）
def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(
            template.format(
                subject=eg.value["email"]["subject"],
                to_email=eg.value["email"]["to"],
                from_email=eg.value["email"]["author"],
                content=eg.value["email"]["email_thread"][:400],  # 截断前 400 字符，避免 prompt 过长
                result=eg.value["label"],
            )
        )
    return "\n\n------------\n\n".join(strs)

In [ ]:
# 模拟一次分诊时的检索：拿一封新邮件去 store 里语义检索最相似的历史案例
email_data = {
        "author": "Sarah Chen <sarah.chen@company.com>",
        "to": "John Doe <john.doe@company.com>",
        "subject": "Update: Backend API Changes Deployed to Staging",
        "email_thread": """Hi John,

    Wanted to let you know that I've deployed the new authentication endpoints we discussed to the staging environment. Key changes include:

    - Implemented JWT refresh token rotation
    - Added rate limiting for login attempts
    - Updated API documentation with new endpoints

    All tests are passing and the changes are ready for review. You can test it out at staging-api.company.com/auth/*

    No immediate action needed from your side - just keeping you in the loop since this affects the systems you're working on.

    Best regards,
    Sarah
    """,
    }
# query=str({"email": email_data})：把整封邮件序列化成字符串作为检索 query，
# limit=1：只取最相似的 1 条历史案例（真实网络请求：需要把 query 转成 embedding）
results = store.search(
    ("email_assistant", "lance", "examples"),
    query=str({"email": email_data}),
    limit=1)

In [ ]:
# 打印格式化后的示例文本，就是最终会被塞进 triage_system_prompt 的 {examples} 部分
print(format_few_shot_examples(results))

In [ ]:
# 这一课把 triage_system_prompt 直接内联在 notebook 里（而不是从 prompts.py 导入），
# 内容和 prompts.py 里补的那份完全一致，多了 {examples} 占位符用来承载上面检索到的历史案例
triage_system_prompt = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Background >
{user_profile_background}.
</ Background >

< Instructions >

{name} gets lots of emails. Your job is to categorize each email into one of three categories:

1. IGNORE - Emails that are not worth responding to or tracking
2. NOTIFY - Important information that {name} should know about but doesn't require a response
3. RESPOND - Emails that need a direct response from {name}

Classify the below email into one of these categories.

</ Instructions >

< Rules >
Emails that are not worth responding to:
{triage_no}

There are also other things that {name} should know about, but don't require an email response. For these, you should notify {name} (using the `notify` response). Examples of this include:
{triage_notify}

Emails that are worth responding to:
{triage_email}
</ Rules >

< Few shot examples >

Here are some examples of previous emails, and how they should be handled.
Follow these examples more than any instructions above

{examples}
</ Few shot examples >
"""

## Setup Routing Node

In [ ]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

In [ ]:
llm = init_chat_model("openai:gpt-4o-mini")

In [ ]:
class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

In [ ]:
llm_router = llm.with_structured_output(Router)

In [ ]:
# triage_system_prompt 这一课改为在上面内联定义了，这里只需要从 prompts.py 导入 triage_user_prompt
from prompts import triage_user_prompt

### Setup Triage Router Node

In [ ]:
from langgraph.graph import add_messages

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

In [ ]:
# triage_router：这一课的核心变化——函数签名从 (state) 变成了 (state, config, store)。
# LangGraph 会按参数名自动注入：config 是运行时配置（里面带 langgraph_user_id），
# store 是编译图时传入的长期记忆存储（见下面 compile(store=store)）。
# 有了 store 就能在分诊之前，先检索这个用户过往的历史分类案例，做成 few-shot 示例。
def triage_router(state: State, config, store) -> Command[
    Literal["response_agent", "__end__"]
]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    # 每个用户的历史案例存在各自的命名空间下，靠 langgraph_user_id 区分（用户间记忆隔离）
    namespace = (
        "email_assistant",
        config['configurable']['langgraph_user_id'],
        "examples"
    )
    # 语义检索：拿"当前这封邮件"去匹配最相似的历史案例（不限定 limit，默认取若干条）
    examples = store.search(
        namespace,
        query=str({"email": state['email_input']})
    )
    examples=format_few_shot_examples(examples)

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
        examples=examples   # 把检索到的历史案例真正塞进 prompt
    )
    user_prompt = triage_user_prompt.format(
        author=author,
        to=to,
        subject=subject,
        email_thread=email_thread
    )
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Respond to the email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        update = None
        goto = END
    elif result.classification == "notify":
        # If real life, this would do something else
        print("🔔 Classification: NOTIFY - This email contains important information")
        update = None
        goto = END
    else:
        raise ValueError(f"Invalid classification: {result.classification}")
    return Command(goto=goto, update=update)

### Setup the rest of the agent


In [ ]:
from langchain_core.tools import tool

In [ ]:
# 工具 1：发送邮件（占位实现）
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    # Placeholder response - in real app would send email
    return f"Email sent to {to} with subject '{subject}'"


In [ ]:
# 工具 2：安排会议（占位实现）
@tool
def schedule_meeting(
    attendees: list[str],
    subject: str,
    duration_minutes: int,
    preferred_day: str
) -> str:
    """Schedule a calendar meeting."""
    # Placeholder response - in real app would check calendar and schedule
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


In [ ]:
# 工具 3：查看日历可用时段（占位实现）
@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # Placeholder response - in real app would check actual calendar
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [ ]:
# 语义记忆工具（同 lesson_3），让 response_agent 也能存取用户相关的事实性记忆
from langmem import create_manage_memory_tool, create_search_memory_tool

In [ ]:
# 注意：命名空间末尾是 "collection"，和上面情景记忆用的 "examples" 是两个不同的命名空间
manage_memory_tool = create_manage_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)
search_memory_tool = create_search_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)

In [ ]:
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:

1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc. in memory for future reference
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""

In [ ]:
def create_prompt(state):
    return [
        {
            "role": "system",
            "content": agent_system_prompt_memory.format(
                instructions=prompt_instructions["agent_instructions"],
                **profile
            )
        }
    ] + state['messages']

In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
# 这里模型换回了 "openai:gpt-4o"（lesson_3 用的是 "anthropic:claude-sonnet-4-6"）
tools= [
    write_email,
    schedule_meeting,
    check_calendar_availability,
    manage_memory_tool,
    search_memory_tool
]
response_agent = create_react_agent(
    "openai:gpt-4o",
    tools=tools,
    prompt=create_prompt,
    # Use this to ensure the store is passed to the agent
    store=store
)

In [ ]:
config = {"configurable": {"langgraph_user_id": "lance"}}

## Build the email agent graph


In [ ]:
email_agent = StateGraph(State)
email_agent = email_agent.add_node(triage_router)
email_agent = email_agent.add_node("response_agent", response_agent)
email_agent = email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile(store=store)

In [ ]:
# 接下来演示：同一个用户（harrison）第一次遇到"想买文档"这类邮件时，
# 因为 store 里还没有他的历史案例，triage_router 检索不到任何 few-shot 示例，
# 分类结果完全靠 triage_system_prompt 里的通用规则判断
email_input = {
    "author": "Tom Jones <tome.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentation?""",
}

In [ ]:
# 用 langgraph_user_id="harrison" 调用，注意这是一个和 "lance" 完全独立的记忆命名空间
response = email_agent.invoke(
    {"email_input": email_input},
    config={"configurable": {"langgraph_user_id": "harrison"}}
)

In [ ]:
# 现在人工给 harrison 补一条"这类邮件应该 ignore"的历史案例（模拟用户手动纠正 agent 的行为）
data = {
    "email": {
    "author": "Tom Jones <tome.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentation?""",
},
    "label": "ignore"
}

In [ ]:
# 注意命名空间第二段是 "harrison"，专属于这个用户的情景记忆
store.put(
    ("email_assistant", "harrison", "examples"),
    str(uuid.uuid4()),
    data
)

In [ ]:
# 再发一次一模一样的邮件，这次 triage_router 应该能检索到刚才存的 "ignore" 案例，分类行为随之改变
email_input = {
    "author": "Tom Jones <tome.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentation?""",
}

In [ ]:
response = email_agent.invoke(
    {"email_input": email_input},
    config={"configurable": {"langgraph_user_id": "harrison"}}
)

In [ ]:
# 换一封内容相似但不完全相同的邮件，验证语义检索是否能泛化匹配（而不是要求逐字相同）
email_input = {
    "author": "Jim Jones <jim.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentation?????""",
}

In [ ]:
# 仍用 harrison 的身份调用：预期能检索到之前存的相似案例，行为保持一致
response = email_agent.invoke(
    {"email_input": email_input},
    config={"configurable": {"langgraph_user_id": "harrison"}}
)

In [ ]:
# 换成 langgraph_user_id="andrew"（一个从没存过任何案例的新用户）：
# 预期 store 检索不到任何历史示例，分类行为回到"无 few-shot 示例"的默认判断，
# 用来验证记忆确实是按用户隔离的，不会被 harrison 的纠正"污染"
response = email_agent.invoke(
    {"email_input": email_input},
    config={"configurable": {"langgraph_user_id": "andrew"}}
)